In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parents[1]
DATA_RAW = ROOT / "data_raw"
DATA_PROCESSED = ROOT / "data_processed"

INPUT_PATH = DATA_RAW / "matches_points_merged" / "merged_WTA_QF+_raw.parquet"
assert INPUT_PATH.exists(), f"Missing input: {INPUT_PATH}"

df = pd.read_parquet(INPUT_PATH)

print("Loaded:", INPUT_PATH)
print("Shape:", df.shape)
print("Unique matches:", df["match_id"].nunique())


Loaded: /Users/leventezsiga/Documents/VU/Thesis_P2-P3/tennis-pressure/data_raw/matches_points_merged/merged_WTA_QF+_raw.parquet
Shape: (48211, 23)
Unique matches: 336


Add columns: "p1_pts_game_after", "p2_pts_game_after", "p1_pts_game_before", "p2_pts_game_before" to keep track of the score of the current game.

In [2]:
df = df.sort_values(["match_id", "SetNo", "GameNo", "PointNumber"]).reset_index(drop=True)

In [ ]:
p1_point = (df["PointWinner"] == 1).astype("int64")
p2_point = (df["PointWinner"] == 2).astype("int64")

grp = df.groupby(["match_id", "SetNo", "GameNo"], sort=False)

df["p1_pts_game_after"] = grp[p1_point.name].cumsum() if p1_point.name in df.columns else grp.cumcount()
df["p2_pts_game_after"] = grp[p2_point.name].cumsum() if p2_point.name in df.columns else grp.cumcount()

In [4]:
df["_p1_point"] = (df["PointWinner"] == 1).astype("int64")
df["_p2_point"] = (df["PointWinner"] == 2).astype("int64")

grp = df.groupby(["match_id", "SetNo", "GameNo"], sort=False)

df["p1_pts_game_after"] = grp["_p1_point"].cumsum()
df["p2_pts_game_after"] = grp["_p2_point"].cumsum()

df["p1_pts_game_before"] = grp["p1_pts_game_after"].shift(1, fill_value=0)
df["p2_pts_game_before"] = grp["p2_pts_game_after"].shift(1, fill_value=0)

df = df.drop(columns=["_p1_point", "_p2_point"])


In [6]:
sample_match = df["match_id"].iloc[0]
sample = df[df["match_id"] == sample_match].head(5)[
    ["match_id","SetNo","GameNo","PointNumber","PointWinner",
     "p1_pts_game_before","p2_pts_game_before","p1_pts_game_after","p2_pts_game_after"]
]
sample

,match_id,SetNo,GameNo,PointNumber,PointWinner,p1_pts_game_before,p2_pts_game_before,p1_pts_game_after,p2_pts_game_after
0,2011-ausopen-2501,1,1,0,0,0,0,0,0
1,2011-ausopen-2501,1,1,1,2,0,0,0,1
2,2011-ausopen-2501,1,1,2,1,0,1,1,1
3,2011-ausopen-2501,1,1,3,1,1,1,2,1
4,2011-ausopen-2501,1,1,4,2,2,1,2,2


Add column "game_score_str". (Classic tennis scoring format)

In [7]:
score_map = {0: "0", 1: "15", 2: "30", 3: "40"}

def format_game_score(r):
    a = r["p1_pts_game_before"]
    b = r["p2_pts_game_before"]

    if pd.isna(a) or pd.isna(b):
        return None

    a, b = int(a), int(b)

    if a >= 3 and b >= 3:
        if a == b:
            return "40–40"
        if a == b + 1:
            return "Ad P1"
        if b == a + 1:
            return "Ad P2"

    return f"{score_map.get(a, '40+')}-{score_map.get(b, '40+')}"


In [8]:
df["game_score_str"] = df.apply(format_game_score, axis=1)

In [9]:
df[[
    "p1_pts_game_before",
    "p2_pts_game_before",
    "game_score_str"
]].head(20)


,p1_pts_game_before,p2_pts_game_before,game_score_str
0,0,0,0-0
1,0,0,0-0
2,0,1,0-15
3,1,1,15-15
4,2,1,30-15
5,2,2,30-30
6,2,3,30-40
7,0,0,0-0
8,0,1,0-15
9,1,1,15-15


Add columns to keep track of sets won by each player within current match. 'P1SetsWon' and 'P2SetsWon'

In [14]:
import numpy as np

def add_set_scores(match: pd.DataFrame) -> pd.DataFrame:
    m = match.copy()

    m["SetNo"] = pd.to_numeric(m["SetNo"], errors="coerce")
    m["P1GamesWon"] = pd.to_numeric(m["P1GamesWon"], errors="coerce")
    m["P2GamesWon"] = pd.to_numeric(m["P2GamesWon"], errors="coerce")

    set_max = m.groupby("SetNo")[["P1GamesWon", "P2GamesWon"]].max()

    p1_gt = (set_max["P1GamesWon"] > set_max["P2GamesWon"]).fillna(False)
    p2_gt = (set_max["P2GamesWon"] > set_max["P1GamesWon"]).fillna(False)

    set_winners = np.where(p1_gt, 1, np.where(p2_gt, 2, np.nan))
    set_winners = pd.Series(set_winners, index=set_max.index)

    p1_sets, p2_sets = [], []
    for _, row in m.iterrows():
        finished = set_winners[set_winners.index < row["SetNo"]]
        p1_sets.append((finished == 1).sum())
        p2_sets.append((finished == 2).sum())

    m["P1SetsWon"] = pd.Series(p1_sets, index=m.index).astype("int64")
    m["P2SetsWon"] = pd.Series(p2_sets, index=m.index).astype("int64")

    return m

df = df.groupby("match_id", group_keys=False).apply(add_set_scores)

/var/folders/sk/00rkx0hd027cbgk6d5rxrl7w0000gn/T/ipykernel_17515/3317290725.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("match_id", group_keys=False).apply(add_set_scores)


In [17]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

In [20]:
test = df[df["match_id"] == df["match_id"].iloc[0]][
    ["match_id","SetNo","GameNo","PointNumber","P1GamesWon","P2GamesWon","P1SetsWon","P2SetsWon"]
].head(70)

test

,match_id,SetNo,GameNo,PointNumber,P1GamesWon,P2GamesWon,P1SetsWon,P2SetsWon
0,2011-ausopen-2501,1,1,0,0,0,0,0
1,2011-ausopen-2501,1,1,1,0,0,0,0
2,2011-ausopen-2501,1,1,2,0,0,0,0
3,2011-ausopen-2501,1,1,3,0,0,0,0
4,2011-ausopen-2501,1,1,4,0,0,0,0
...,...,...,...,...,...,...,...,...
65,2011-ausopen-2501,1,9,65,3,5,0,0
66,2011-ausopen-2501,1,9,66,3,5,0,0
67,2011-ausopen-2501,1,9,67,3,5,0,0
68,2011-ausopen-2501,1,9,68,3,6,0,0


In [19]:
pd.reset_option("display.max_rows")
pd.reset_option("display.max_columns")

Add outcome flags for last point of each game, last point of each set, and last point of each match.

In [21]:
def add_outcome_flags(df):
    df = df.copy()

    df["is_game_end"] = (
        df.groupby(["match_id", "SetNo", "GameNo"])["PointNumber"]
          .transform("max") == df["PointNumber"]
    )

    df["is_set_end"] = (
        df.groupby(["match_id", "SetNo"])["PointNumber"]
          .transform("max") == df["PointNumber"]
    )

    df["is_match_end"] = (
        df.groupby("match_id")["PointNumber"]
          .transform("max") == df["PointNumber"]
    )

    return df

df = add_outcome_flags(df)


In [22]:
print("game_end per game (should be 1):")
print(df.groupby(["match_id","SetNo","GameNo"])["is_game_end"].sum().value_counts().head())

print("\nset_end per set (should be 1):")
print(df.groupby(["match_id","SetNo"])["is_set_end"].sum().value_counts().head())

print("\nmatch_end per match (should be 1):")
print(df.groupby(["match_id"])["is_match_end"].sum().value_counts().head())


game_end per game (should be 1):
is_game_end
1    7355
Name: count, dtype: Int64

set_end per set (should be 1):
is_set_end
1    782
Name: count, dtype: Int64

match_end per match (should be 1):
is_match_end
1    336
Name: count, dtype: Int64


Save. Resulting dataset contains columns to keep track of scores throughout all the matches.

In [23]:
OUT_DIR = DATA_PROCESSED
out_path = OUT_DIR / "01_score_state_features.parquet"
df.to_parquet(out_path, index=False)
print("Saved:", out_path)
print("Shape:", df.shape)

Saved: /Users/leventezsiga/Documents/VU/Thesis_P2-P3/tennis-pressure/data_processed/01_score_state_features.parquet
Shape: (48211, 33)
